# CC Approval & AECB Data Pull (2025-06 to 2026-06)

Pull credit card numbers approved between Jun 2025 and Jun 2026 from the credit card history view, then retrieve corresponding AECB bureau data for those cases.

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================================
# Step 1: Pull credit cards approved between Jun 2025 and Jun 2026
# ============================================================================
# df_cc = spark.sql("""
#     SELECT *
#     FROM mis_automation.silver_credit_card.vw_credit_card_history
#     WHERE APPROVEDATE >= '2025-06-01'
#       AND APPROVEDATE <  '2026-07-01'
# """)

# raw_rows = df_cc.count()
# raw_cards = df_cc.select('CREDITCARDNO').distinct().count()
# print(f"Before exclusions — Rows: {raw_rows:,} | Unique cards: {raw_cards:,}")

# # ============================================================================
# # Step 2: Compute per-card OPEN snapshot count (needed for Exclusion 9)
# # ============================================================================
# # w = Window.partitionBy("CREDITCARDNO")
# # df_cc = df_cc.withColumn(
# #     "open_count",
# #     F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w)
# # )

# # ============================================================================
# # Step 3: Apply business exclusion filters (same as EDA_1)
# # ============================================================================
# exclude_cond = (
#     (F.upper(F.trim(F.col("CLASSCODE"))) == "SME") |                                    # Exclusion 1: SME class code
#     (~F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS")) |                  # Exclusion 2: Non-CARDS segment
#     (F.col("VIP") == 1) |                                                              # Exclusion 3a: VIP customers
#     (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y") |                             # Exclusion 3b: Ruling family
#     # (F.col("AECB_DATA").isNull()) |                                                  # Exclusion 4: No AECB data (disabled)
#     # (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) != "SALARIED") |                        # Exclusion 6: Non-salaried
#     (F.col("PRIMARYCARD") == 0) |                                                      # Exclusion 7: Supplementary cards
#     (F.upper(F.trim(F.col("CLASSCODE"))) == "STAFF") |                                  # Exclusion 8: Staff cards
#     # (F.col("open_count") < 3) |                                                        # Exclusion 9: Fewer than 3 OPEN snapshots
#     (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM")# |                        # Exclusion 10a: Salaried-IM
#     # (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
#     #     "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))               # Exclusion 10b: Unwanted segment policies
# )

# df_cc_clean = df_cc.filter(~exclude_cond)#.drop("open_count")

# clean_rows = df_cc_clean.count()
# clean_cards = df_cc_clean.select('CREDITCARDNO').distinct().count()
# clean_refs = df_cc_clean.select('REFERENCE_NUMBER').distinct().count()
# print(f"After exclusions  — Rows: {clean_rows:,} | Unique cards: {clean_cards:,} | Unique refs: {clean_refs:,}")
# print(f"Excluded: {raw_rows - clean_rows:,} rows, {raw_cards - clean_cards:,} cards")

# df_cc_pd = df_cc_clean.toPandas()
# print(f"\nApproval date range: {df_cc_pd['APPROVEDATE'].min()} to {df_cc_pd['APPROVEDATE'].max()}")
# df_cc_pd.head()

# ============================================================================
# Missing-value handling: create missing indicators + impute NaN
# ============================================================================
import numpy as np

# Define imputation rules: (column, missing_indicator_name, fill_value)
IMPUTE_RULES = [
    ('Since_Chq_Rtn_DT',   'Since_Chq_Rtn_DT_is_missing',   9999),
    ('MOB_Since_Chq_Rtn',   'MOB_Since_Chq_Rtn_is_missing',  99),
    ('max_DEFAULT_DT',      'max_DEFAULT_DT_is_missing',      9999),
    ('MOB_Since_DELQ',      'MOB_Since_DELQ_is_missing',      99),
    ('OTHER_MOB_recent',    'OTHER_MOB_recent_is_missing',    -1),
    ('PL_MOB_recent',       'PL_MOB_recent_is_missing',       -1),
    ('Mean_CC_utilization',  'Mean_CC_utilization_is_missing', -1),
    ('Max_CC_utilization',   'Max_CC_utilization_is_missing',  -1),
    ('Min_CC_utilization',   'Min_CC_utilization_is_missing',  -1),
    ('CC_MOB_recent',        'MCC_MOB_recent_is_missing',     -1),
    ('Active_noof_prds',     'Active_noof_prds_is_missing',    0),
    ('Active_dist_prds',     'MActive_dist_prds_is_missing',   0),
    ('first_product',        'first_product_is_missing',       'UNKNOWN'),
]


def treat_missing(df, impute_rules=IMPUTE_RULES, verbose=True):
    """
    Create missing-indicator columns and impute NaN values in-place.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to treat (modified in-place and also returned).
    impute_rules : list of (col, flag_name, fill_value)
        Each tuple defines the column to impute, the name of the
        binary missing-indicator column, and the fill value.
    verbose : bool
        If True, print a summary after imputation.

    Returns
    -------
    pd.DataFrame
        The same DataFrame (modified in-place).
    """
    for col, flag, fill_val in impute_rules:
        df[flag] = df[col].isna().astype(int)
        df[col] = df[col].fillna(fill_val)

    if verbose:
        print(f"{'=' * 70}")
        print(f"Missing-value treatment summary  ({len(df):,} rows)")
        print(f"{'=' * 70}")
        for col, flag, fill_val in impute_rules:
            nan_remaining = df[col].isna().sum()
            missing_1 = (df[flag] == 1).sum()
            missing_0 = (df[flag] == 0).sum()
            print(f"  {col:<25s} NaN remaining: {nan_remaining} | "
                  f"{flag}==1: {missing_1:,} | ==0: {missing_0:,} | fill={fill_val}")
        print()

    return df

In [ ]:
def pull_cc_with_exclusions(approvedate_start: str, approvedate_end: str):
    """
    Pull credit card approvals within [approvedate_start, approvedate_end)
    and apply business exclusion + inclusion filters.

    Parameters
    ----------
    approvedate_start : str  – inclusive lower bound, e.g. '2025-06-01'
    approvedate_end   : str  – exclusive upper bound, e.g. '2026-07-01'

    Returns
    -------
    df_cc_clean : pyspark DataFrame after all exclusions/inclusions
    """
    # ========================================================================
    # Step 1: Pull raw data
    # ========================================================================
    df_cc = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE APPROVEDATE >= '{approvedate_start}'
          AND APPROVEDATE <  '{approvedate_end}'
    """)

    # Force VIP to be nullable across all partitions to prevent ArrowInvalid on .toPandas()
    # (Source view has inconsistent nullability for VIP across partition files)
    df_cc = df_cc.withColumn("VIP", F.when(F.lit(True), F.col("VIP")))

    raw_rows = df_cc.count()
    raw_cards = df_cc.select('CREDITCARDNO').distinct().count()
    print(f"Before exclusions — Rows: {raw_rows:,} | Unique cards: {raw_cards:,}")

    # Monthly approved card count (before exclusions)
    monthly_approved = (
        df_cc
        .withColumn("approve_month", F.date_format(F.col("APPROVEDATE"), "yyyy-MM"))
        .groupBy("approve_month")
        .agg(F.countDistinct("CREDITCARDNO").alias("approved_cards"))
        .orderBy("approve_month")
    )
    print("\nMonthly approved card count (before exclusions):")
    monthly_approved.show(truncate=False)

    # ====================================================================
    # Step 2: Compute per-card OPEN snapshot count (needed for Exclusion 9)
    # ====================================================================
    w = Window.partitionBy("CREDITCARDNO")
    df_cc = df_cc.withColumn(
        "open_count",
        F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w)
    )

    # ====================================================================
    # Step 3: Apply business exclusion filters
    # ====================================================================
    exclude_cond = (
        (F.col("VIP") == "1") |                                                              # Exclusion 3a: VIP customers
        (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y") |                             # Exclusion 3b: Ruling family
        # (F.col("AECB_DATA").isNull()) |                                                  # Exclusion 4: No AECB data (disabled)
        (F.col("PRIMARYCARD") == 0) |                                                      # Exclusion 7: Supplementary cards
        # (F.col("open_count") < 3) |                                                        # Exclusion 9: Fewer than 3 OPEN snapshots
        (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM") |                        # Exclusion 10a: Salaried-IM
        (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
            "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))               # Exclusion 10b: Unwanted segment policies
    )

    inclusive_cond = (
        (F.col("CLASSCODE").isNull() | ~F.trim(F.col("CLASSCODE")).isin("STAFF", "SME"))
        & (F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS"))
        & (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) == "SALARIED")
    )

    df_cc_clean = df_cc.filter(~exclude_cond).filter(inclusive_cond)

    clean_rows = df_cc_clean.count()
    clean_cards = df_cc_clean.select('CREDITCARDNO').distinct().count()
    clean_refs = df_cc_clean.select('REFERENCE_NUMBER').distinct().count()
    print(f"After exclusions  — Rows: {clean_rows:,} | Unique cards: {clean_cards:,} | Unique refs: {clean_refs:,}")
    print(f"Excluded: {raw_rows - clean_rows:,} rows, {raw_cards - clean_cards:,} cards")

    # Monthly approved card count (after exclusions)
    monthly_approved_clean = (
        df_cc_clean
        .withColumn("approve_month", F.date_format(F.col("APPROVEDATE"), "yyyy-MM"))
        .groupBy("approve_month")
        .agg(F.countDistinct("CREDITCARDNO").alias("approved_cards"))
        .orderBy("approve_month")
    )
    print("\nMonthly new approved card count (after exclusions):")
    monthly_approved_clean.show(truncate=False)

    return df_cc_clean


# Example usage
df_cc_pd = pull_cc_with_exclusions('2025-06-01', '2026-07-01').toPandas()

In [ ]:
# ============================================================================
# Step 2: Pull AECB bureau data for matching reference numbers
# ============================================================================

# Get unique reference numbers from the credit card pull
ref_numbers = [str(r) for r in df_cc_pd['REFERENCE_NUMBER'].unique().tolist()]
print(f"Unique reference numbers to look up: {len(ref_numbers):,}")

# Load AECB feature store and filter to matching reference numbers
aecb = spark.sql("""
    SELECT *
    FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store
""")

aecb_filtered = (
    aecb
    .withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
    .filter(F.col("Reference_Number").isin(ref_numbers))
)

aecb_pd = aecb_filtered.toPandas()
print(f"AECB rows retrieved: {aecb_pd.shape[0]:,}")
print(f"AECB columns: {aecb_pd.shape[1]}")
print(f"Unique reference numbers in AECB: {aecb_pd['Reference_Number'].nunique():,}")

# Check coverage: how many CC reference numbers have AECB data
matched = set(aecb_pd['Reference_Number'].astype(str)) & set(ref_numbers)
print(f"\nCoverage: {len(matched):,} / {len(ref_numbers):,} reference numbers matched ({len(matched)/len(ref_numbers):.1%})")
aecb_pd.head()

aecb_pd = treat_missing(aecb_pd)

In [ ]:
# ============================================================================
# Step 3: Merge AECB data with credit card data & filter to Thin File
# ============================================================================

# Ensure join keys are the same dtype
aecb_pd['Reference_Number'] = aecb_pd['Reference_Number'].astype(str)
df_cc_pd['REFERENCE_NUMBER'] = df_cc_pd['REFERENCE_NUMBER'].astype(str)

# Merge on reference number
df_merged = df_cc_pd.merge(
    aecb_pd,
    left_on='REFERENCE_NUMBER',
    right_on='Reference_Number',
    how='inner'
)
print(f"Merged shape: {df_merged.shape}")
print(f"file_type distribution:\n{df_merged['file_type'].value_counts(dropna=False)}")

# Filter to Thin File only
df_thin = df_merged[df_merged['file_type'] == 'Thin File'].reset_index(drop=True)
print(f"\nThin File shape: {df_thin.shape}")
print(f"Unique cards: {df_thin['CREDITCARDNO'].nunique():,}")
print(f"Unique refs:  {df_thin['REFERENCE_NUMBER'].nunique():,}")
df_thin.head()

In [ ]:
# ============================================================================
# Step 4: Bad rate (90 DPD) by snapshot month
# Use df_thin (Thin File universe)
# ============================================================================

# Ensure DELINQUENCYBUCKET is numeric
df_thin['DELINQUENCYBUCKET'] = pd.to_numeric(df_thin['DELINQUENCYBUCKET'], errors='coerce')

# Flag bad per row: DELINQUENCYBUCKET > 3 → 90 DPD
df_thin['is_bad'] = (df_thin['DELINQUENCYBUCKET'] > 3).astype(int)
df_thin['snapshot_month'] = pd.to_datetime(df_thin['SNAPSHOT_DATE']).dt.to_period('M')

# Per card × snapshot month: flag if bad at that snapshot
snap_bad = (
    df_thin
    .groupby(['CREDITCARDNO', 'snapshot_month'], as_index=False)
    .agg(is_bad=('is_bad', 'max'))
)

# Aggregate by snapshot month: total cards and bad cards
bad_by_snap = (
    snap_bad
    .groupby('snapshot_month', as_index=False)
    .agg(
        total_cards=('CREDITCARDNO', 'nunique'),
        bad_cards=('is_bad', 'sum')
    )
)
bad_by_snap['bad_rate'] = bad_by_snap['bad_cards'] / bad_by_snap['total_cards'] * 100
bad_by_snap = bad_by_snap.sort_values('snapshot_month')

print("Bad rate (90 DPD) by snapshot month:")
print(bad_by_snap.to_string(index=False))
print(f"\nOverall bad rate: {bad_by_snap['bad_cards'].sum() / bad_by_snap['total_cards'].sum():.4%}")
display(bad_by_snap)

In [ ]:
# ============================================================================
# Cumulative bad rate by approval-month vintage across snapshot months
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Ensure helper columns exist
df_thin['approve_month'] = pd.to_datetime(df_thin['APPROVEDATE']).dt.to_period('M')
df_thin['snapshot_month'] = pd.to_datetime(df_thin['SNAPSHOT_DATE']).dt.to_period('M')

# Per card × snapshot month: flag if bad at that snapshot
snap_vintage = (
    df_thin
    .groupby(['CREDITCARDNO', 'approve_month', 'snapshot_month'], as_index=False)
    .agg(is_bad=('is_bad', 'max'))
)

# Aggregate: bad rate per vintage × snapshot month
vintage_snap = (
    snap_vintage
    .groupby(['approve_month', 'snapshot_month'], as_index=False)
    .agg(
        total_cards=('CREDITCARDNO', 'nunique'),
        bad_cards=('is_bad', 'sum')
    )
)
vintage_snap['bad_rate'] = vintage_snap['bad_cards'] / vintage_snap['total_cards'] * 100
vintage_snap = vintage_snap.sort_values(['approve_month', 'snapshot_month'])

# --- Plot: one line per vintage -----------------------------------------------
fig, ax = plt.subplots(figsize=(16, 7))

for vintage, grp in vintage_snap.groupby('approve_month'):
    grp = grp.sort_values('snapshot_month')
    ax.plot(
        grp['snapshot_month'].astype(str),
        grp['bad_rate'],
        marker='o', markersize=4, linewidth=1.8,
        label=str(vintage)
    )

ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('90 DPD Bad Rate by Approval-Month Vintage over Snapshot Months', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f%%'))
ax.legend(title='Approval Month', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Plot 1: Overall bad rate by snapshot month  (from vintage_snap)
# Plot 2: Bad rate by snapshot month × approval-month vintage (heatmap)
# ============================================================================
import seaborn as sns
#applciation source code 
# --- Plot 1: Overall bad rate by snapshot month -------------------------------
overall = (
    vintage_snap
    .groupby('snapshot_month', as_index=False)
    .agg(total_cards=('total_cards', 'sum'),
         bad_cards=('bad_cards', 'sum'))
)
overall['bad_rate'] = overall['bad_cards'] / overall['total_cards'] * 100
overall = overall.sort_values('snapshot_month')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(overall['snapshot_month'].astype(str), overall['bad_rate'],
        marker='o', color='crimson', linewidth=2)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('Overall 90 DPD Bad Rate by Snapshot Month', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f%%'))
plt.xticks(rotation=45, ha='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot 2: Heatmap – bad rate by vintage × snapshot month -------------------
pivot = vintage_snap.pivot(
    index='approve_month', columns='snapshot_month', values='bad_rate'
)
pivot.index = pivot.index.astype(str)
pivot.columns = pivot.columns.astype(str)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Bad Rate %'}, ax=ax)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Approval Month Vintage', fontsize=12)
ax.set_title('90 DPD Bad Rate (%) by Vintage × Snapshot Month', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# PRIOR YEAR: Pull credit cards approved between Jun 2024 and Jun 2025
# ============================================================================
df_cc_pd_prev = pull_cc_with_exclusions(approvedate_start='2024-06-01', approvedate_end='2025-07-01').toPandas()

df_cc_pd_prev.head()

In [ ]:
df_cc_pd_prev.head(2)

In [ ]:
# ============================================================================
# PRIOR YEAR: Pull AECB, merge, filter Thin File
# ============================================================================
ref_numbers_prev = [str(r) for r in df_cc_pd_prev['REFERENCE_NUMBER'].unique().tolist()]
print(f"Unique reference numbers to look up: {len(ref_numbers_prev):,}")

aecb_filtered_prev = (
    aecb
    .withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
    .filter(F.col("Reference_Number").isin(ref_numbers_prev))
)
aecb_pd_prev = aecb_filtered_prev.toPandas()
print(f"AECB rows retrieved: {aecb_pd_prev.shape[0]:,}")
print(f"Unique reference numbers in AECB: {aecb_pd_prev['Reference_Number'].nunique():,}")

aecb_pd_prev = treat_missing(aecb_pd_prev)

# Merge
aecb_pd_prev['Reference_Number'] = aecb_pd_prev['Reference_Number'].astype(str)
df_cc_pd_prev['REFERENCE_NUMBER'] = df_cc_pd_prev['REFERENCE_NUMBER'].astype(str)

df_merged_prev = df_cc_pd_prev.merge(
    aecb_pd_prev, left_on='REFERENCE_NUMBER', right_on='Reference_Number', how='inner'
)
print(f"\nMerged shape: {df_merged_prev.shape}")
print(f"file_type distribution:\n{df_merged_prev['file_type'].value_counts(dropna=False)}")

df_thin_prev = df_merged_prev[df_merged_prev['file_type'] == 'Thin File'].reset_index(drop=True)
print(f"\nThin File shape: {df_thin_prev.shape}")
print(f"Unique cards: {df_thin_prev['CREDITCARDNO'].nunique():,}")

In [ ]:
aecb_pd_prev.columns

In [ ]:
# ============================================================================
# PRIOR YEAR: Define 90 DPD target, bad rate analysis, vintage heatmap
# (Same exercise as 2025-06 to 2026-06 dataset)
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Ensure DELINQUENCYBUCKET is numeric
df_thin_prev['DELINQUENCYBUCKET'] = pd.to_numeric(df_thin_prev['DELINQUENCYBUCKET'], errors='coerce')

# Flag bad per row: DELINQUENCYBUCKET > 3 → 90 DPD
df_thin_prev['is_bad'] = (df_thin_prev['DELINQUENCYBUCKET'] > 3).astype(int)
df_thin_prev['snapshot_month'] = pd.to_datetime(df_thin_prev['SNAPSHOT_DATE']).dt.to_period('M')
df_thin_prev['approve_month'] = pd.to_datetime(df_thin_prev['APPROVEDATE']).dt.to_period('M')

# Per card × snapshot month: flag if bad at that snapshot
snap_bad_prev = (
    df_thin_prev
    .groupby(['CREDITCARDNO', 'snapshot_month'], as_index=False)
    .agg(is_bad=('is_bad', 'max'))
)

# Aggregate by snapshot month: total cards and bad cards
bad_by_snap_prev = (
    snap_bad_prev
    .groupby('snapshot_month', as_index=False)
    .agg(
        total_cards=('CREDITCARDNO', 'nunique'),
        bad_cards=('is_bad', 'sum')
    )
)
bad_by_snap_prev['bad_rate'] = bad_by_snap_prev['bad_cards'] / bad_by_snap_prev['total_cards'] * 100
bad_by_snap_prev = bad_by_snap_prev.sort_values('snapshot_month')

print("Bad rate (90 DPD) by snapshot month (2024-2025):")
print(bad_by_snap_prev.to_string(index=False))
print(f"\nOverall bad rate: {bad_by_snap_prev['bad_cards'].sum() / bad_by_snap_prev['total_cards'].sum():.4%}")

# ============================================================================
# Per card × vintage × snapshot month: flag if bad
# ============================================================================
snap_vintage_prev = (
    df_thin_prev
    .groupby(['CREDITCARDNO', 'approve_month', 'snapshot_month'], as_index=False)
    .agg(is_bad=('is_bad', 'max'))
)

vintage_snap_prev = (
    snap_vintage_prev
    .groupby(['approve_month', 'snapshot_month'], as_index=False)
    .agg(
        total_cards=('CREDITCARDNO', 'nunique'),
        bad_cards=('is_bad', 'sum')
    )
)
vintage_snap_prev['bad_rate'] = vintage_snap_prev['bad_cards'] / vintage_snap_prev['total_cards'] * 100
vintage_snap_prev = vintage_snap_prev.sort_values(['approve_month', 'snapshot_month'])

# ============================================================================
# Plot 1: Vintage lines – one line per approval month
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))

for vintage, grp in vintage_snap_prev.groupby('approve_month'):
    grp = grp.sort_values('snapshot_month')
    ax.plot(
        grp['snapshot_month'].astype(str),
        grp['bad_rate'],
        marker='o', markersize=4, linewidth=1.8,
        label=str(vintage)
    )

ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('90 DPD Bad Rate by Approval-Month Vintage over Snapshot Months (2024-2025)', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f%%'))
ax.legend(title='Approval Month', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ============================================================================
# Plot 2: Overall bad rate by snapshot month
# ============================================================================
overall_prev = (
    vintage_snap_prev
    .groupby('snapshot_month', as_index=False)
    .agg(total_cards=('total_cards', 'sum'),
         bad_cards=('bad_cards', 'sum'))
)
overall_prev['bad_rate'] = overall_prev['bad_cards'] / overall_prev['total_cards'] * 100
overall_prev = overall_prev.sort_values('snapshot_month')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(overall_prev['snapshot_month'].astype(str), overall_prev['bad_rate'],
        marker='o', color='crimson', linewidth=2)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('Overall 90 DPD Bad Rate by Snapshot Month (2024-2025)', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f%%'))
plt.xticks(rotation=45, ha='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# Plot 3: Heatmap – bad rate by vintage × snapshot month
# ============================================================================
pivot_prev = vintage_snap_prev.pivot(
    index='approve_month', columns='snapshot_month', values='bad_rate'
)
pivot_prev.index = pivot_prev.index.astype(str)
pivot_prev.columns = pivot_prev.columns.astype(str)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot_prev, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Bad Rate %'}, ax=ax)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Approval Month Vintage', fontsize=12)
ax.set_title('90 DPD Bad Rate (%) by Vintage × Snapshot Month (2024-2025)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Compare AECB features of bad cases at two time windows
#   Group 0: Bad at snapshot 2025-06/07, approved 2024-06 to 2025-07
#   Group 1: Bad at snapshot 2026-06/07, approved 2025-06 to 2026-07
# ============================================================================
import matplotlib.pyplot as plt

# --- Helper columns -----------------------------------------------------------
# Use prior-year data for Group 0 and current-year data for Group 1
df_thin_prev['snapshot_month_str'] = pd.to_datetime(df_thin_prev['SNAPSHOT_DATE']).dt.to_period('M').astype(str)
df_thin_prev['approve_month_str'] = pd.to_datetime(df_thin_prev['APPROVEDATE']).dt.to_period('M').astype(str)

df_thin['snapshot_month_str'] = pd.to_datetime(df_thin['SNAPSHOT_DATE']).dt.to_period('M').astype(str)
df_thin['approve_month_str'] = pd.to_datetime(df_thin['APPROVEDATE']).dt.to_period('M').astype(str)

# --- Group 0: bad at 2025-06/07, approved between 2024-06 and 2025-07 ---------
bad_prev = df_thin_prev[df_thin_prev['is_bad'] == 1].copy()
bad_group0_cards = bad_prev[
    bad_prev['snapshot_month_str'].isin(['2025-06', '2025-07']) &
    (bad_prev['approve_month_str'] >= '2024-06') &
    (bad_prev['approve_month_str'] < '2025-07')
]['CREDITCARDNO'].unique()

# --- Group 1: bad at 2026-06/07, approved between 2025-06 and 2026-07 ---------
bad_curr = df_thin[df_thin['is_bad'] == 1].copy()
bad_group1_cards = bad_curr[
    bad_curr['snapshot_month_str'].isin(['2026-06', '2026-07']) &
    (bad_curr['approve_month_str'] >= '2025-06') &
    (bad_curr['approve_month_str'] < '2026-07')
]['CREDITCARDNO'].unique()

print(f"Group 0 (bad @ 2025-06/07, approved 2024-06→2025-07): {len(bad_group0_cards):,} unique cards")
print(f"Group 1 (bad @ 2026-06/07, approved 2025-06→2026-07): {len(bad_group1_cards):,} unique cards")

In [ ]:
# ============================================================================
# Train a Decision Tree to distinguish Group 0 (bad_prev) vs Group 1 (bad_curr)
# Uses only meaningful AECB / CC numeric features — no IDs, dates, or labels
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, accuracy_score
)
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# --- 1. Build labelled dataset ------------------------------------------------
bad_prev_df = bad_prev.copy()
bad_prev_df['label'] = 0   # Group 0
bad_curr_df = bad_curr.copy()
bad_curr_df['label'] = 1   # Group 1

combined = pd.concat([bad_prev_df, bad_curr_df], ignore_index=True)
print(f"Combined shape: {combined.shape}")
print(f"Label distribution:\n{combined['label'].value_counts()}")

# --- 2. Feature selection: drop identity, date, and target columns -----------
drop_keywords = [
    'date', 'DATE', 'CREDITCARDNO', 'REFERENCE_NUMBER', 'Reference_Number',
    'CARDNUMBER', 'ACCOUNTNUMBER', 'CIFNO', 'CIF', 'CARDNO',
    'snapshot_month', 'approve_month', 'month_str', 'label',
    'is_bad', 'group', 'group_label', 'file_type', 'DELINQUENCYBUCKET',
    'SNAPSHOT_DATE', 'APPROVEDATE', 'APPLICATION_DATE', 'OPEN_DATE',
    'CLOSE_DATE', 'LAST_PAYMENT_DATE', 'FIRST_PAYMENT_DATE',
    'snapshot_month_str', 'approve_month_str',
    'PRODUCT', 'BRANCH', 'NATIONALITY', 'GENDER', 'STATUS',
    'CARD_TYPE', 'CATEGORY', 'SEGMENT'
]

cols_to_drop = [c for c in combined.columns
                if any(kw.lower() in c.lower() for kw in drop_keywords)]

# Keep only numeric columns after dropping unwanted ones
feature_df = combined.drop(columns=[c for c in cols_to_drop if c in combined.columns], errors='ignore')
feature_df = feature_df.select_dtypes(include='number')

# Drop columns that are all NaN or have zero variance
feature_df = feature_df.dropna(axis=1, how='all')
feature_df = feature_df.loc[:, feature_df.nunique() > 1]

feature_cols = feature_df.columns.tolist()
print(f"\nFeatures used ({len(feature_cols)}): {feature_cols}")

X = feature_df.values
y = combined['label'].values

# --- 3. Impute remaining NaNs with column median -----------------------------
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# --- 4. Train / Test split (stratified) --------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train label dist: {np.bincount(y_train)}")
print(f"Test  label dist: {np.bincount(y_test)}")

# --- 5. Train a simple Decision Tree -----------------------------------------
mlflow.sklearn.autolog(log_models=False)   # log params & metrics automatically

with mlflow.start_run(run_name="DT_bad_prev_vs_bad_curr") as run:
    dt = DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=20, random_state=42, class_weight='balanced'
    )
    dt.fit(X_train, y_train)

    y_pred = dt.predict(X_test)
    y_prob = dt.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)

    mlflow.log_metric("test_accuracy", acc)
    mlflow.log_metric("test_roc_auc", auc)

    # Log the model
    sig = infer_signature(
        pd.DataFrame(X_train, columns=feature_cols),
        dt.predict(X_train)
    )
    mlflow.sklearn.log_model(
        dt, name="decision_tree_bad_group_classifier",
        signature=sig,
        input_example=pd.DataFrame(X_train[:3], columns=feature_cols)
    )

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT  (Test Set)")
print("=" * 80)
print(f"Accuracy : {acc:.4f}")
print(f"ROC AUC  : {auc:.4f}")
print(classification_report(y_test, y_pred,
                            target_names=['G0: Bad@2025', 'G1: Bad@2026']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# --- 6. Feature Importance ---------------------------------------------------
importances = dt.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, max(6, len(feat_imp[feat_imp['importance'] > 0]) * 0.35)))
top_feats = feat_imp[feat_imp['importance'] > 0]
ax.barh(top_feats['feature'], top_feats['importance'], color='#3498db')
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Decision Tree — Feature Importances\n(distinguishing Bad@2025 vs Bad@2026)', fontsize=13)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# --- 7. Tree Visualisation ---------------------------------------------------
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(dt, feature_names=feature_cols, class_names=['G0', 'G1'],
          filled=True, rounded=True, fontsize=8, ax=ax,
          max_depth=3, impurity=False, proportion=True)
ax.set_title('Decision Tree (top 3 levels)', fontsize=14)
plt.tight_layout()
plt.show()

# --- 8. Interpretation -------------------------------------------------------
print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)
if auc > 0.65:
    print(f"✅ ROC AUC = {auc:.3f} — the model CAN distinguish the two bad groups.")
    print("   The two cohorts of bad customers are meaningfully DIFFERENT.")
    print("   Top discriminating features:")
    for _, row in feat_imp.head(5).iterrows():
        if row['importance'] > 0:
            print(f"     • {row['feature']}  (importance = {row['importance']:.4f})")
elif auc > 0.55:
    print(f"⚠️  ROC AUC = {auc:.3f} — there are SLIGHT differences between the groups.")
    print("   Some features differ, but the groups are largely similar.")
else:
    print(f"❌ ROC AUC = {auc:.3f} — the model CANNOT distinguish the two groups.")
    print("   The two bad cohorts are essentially the SAME in terms of AECB/CC features.")

In [ ]:
# ============================================================================
# Count of newly approved cards in each time window
# ============================================================================

# Group 0: approved 2024-06 → 2025-07 (prior year)
new_cards_prev = df_thin_prev[
    (pd.to_datetime(df_thin_prev['APPROVEDATE']) >= '2024-06-01') &
    (pd.to_datetime(df_thin_prev['APPROVEDATE']) < '2025-07-01')
]['CREDITCARDNO'].nunique()

# Group 1: approved 2025-06 → 2026-07 (current year)
new_cards_curr = df_thin[
    (pd.to_datetime(df_thin['APPROVEDATE']) >= '2025-06-01') &
    (pd.to_datetime(df_thin['APPROVEDATE']) < '2026-07-01')
]['CREDITCARDNO'].nunique()

print(f"New cards approved 2024-06 → 2025-07: {new_cards_prev:,}")
print(f"New cards approved 2025-06 → 2026-07: {new_cards_curr:,}")
print(f"Change: {new_cards_curr - new_cards_prev:,} ({(new_cards_curr - new_cards_prev) / new_cards_prev * 100:+.2f}%)")

# ============================================================================
# Plot: Approved cards by month (2024-06 to 2026-06)
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Combine both datasets and compute approve month
df_prev_tmp = df_thin_prev[['CREDITCARDNO', 'APPROVEDATE']].copy()
df_curr_tmp = df_thin[['CREDITCARDNO', 'APPROVEDATE']].copy()
df_all = pd.concat([df_prev_tmp, df_curr_tmp], ignore_index=True)
df_all['approve_month'] = pd.to_datetime(df_all['APPROVEDATE']).dt.to_period('M')

# Filter to 2024-06 through 2026-06
df_all = df_all[
    (df_all['approve_month'] >= pd.Period('2024-06', 'M')) &
    (df_all['approve_month'] <= pd.Period('2026-06', 'M'))
]

# Count unique cards per approval month
cards_by_month = (
    df_all
    .groupby('approve_month', as_index=False)
    .agg(approved_cards=('CREDITCARDNO', 'nunique'))
    .sort_values('approve_month')
)
cards_by_month['month_str'] = cards_by_month['approve_month'].astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(cards_by_month['month_str'], cards_by_month['approved_cards'], color='#3498db', edgecolor='white')
ax.plot(cards_by_month['month_str'], cards_by_month['approved_cards'], marker='o', color='#e74c3c', linewidth=1.5, markersize=5)

# Annotate each bar with the count
for i, row in cards_by_month.iterrows():
    ax.text(row['month_str'], row['approved_cards'] + max(cards_by_month['approved_cards']) * 0.01,
            f"{row['approved_cards']:,}", ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xlabel('Approval Month', fontsize=12)
ax.set_ylabel('Number of Approved Cards', fontsize=12)
ax.set_title('Newly Approved Credit Cards by Month (2024-06 to 2026-06)', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Check NaN counts per feature in aecb_pd and aecb_pd_prev
# ============================================================================
# import pandas as pd

# for name, df in [('aecb_pd', aecb_pd), ('aecb_pd_prev', aecb_pd_prev)]:
#     na_counts = df.isna().sum()
#     na_counts = na_counts[na_counts > 0].sort_values(ascending=False)
#     na_pct = (na_counts / len(df) * 100).round(2)

#     summary = pd.DataFrame({
#         'na_count': na_counts,
#         'na_pct (%)': na_pct,
#         'total_rows': len(df)
#     })

#     print(f"{'=' * 80}")
#     print(f"{name}: {len(df):,} rows | {df.shape[1]} columns | "
#           f"{len(na_counts)} features with NaN values")
#     print(f"{'=' * 80}")
#     if summary.empty:
#         print("No columns with NaN values.")
#     else:
#         for col_name, row in summary.iterrows():
#             print(f"  {col_name}: {int(row['na_count']):,} NaN ({row['na_pct (%)']:.2f}%)")
#     print()

In [ ]:
# aecb_pd.Active_noof_prds.value_counts()

In [ ]:
# ============================================================================
# Pull bad cases from both periods, label Group 0 (early) / Group 1 (latest),
# join AECB + CC features, and compare what changed
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Map bad cards → REFERENCE_NUMBER → AECB features ---------------------
# Group 0: map card -> ref from prior-year data
card_to_ref_prev = df_thin_prev[['CREDITCARDNO', 'REFERENCE_NUMBER']].drop_duplicates()
refs_group0 = card_to_ref_prev[card_to_ref_prev['CREDITCARDNO'].isin(bad_group0_cards)]['REFERENCE_NUMBER'].unique()

# Group 1: map card -> ref from current-year data
card_to_ref_curr = df_thin[['CREDITCARDNO', 'REFERENCE_NUMBER']].drop_duplicates()
refs_group1 = card_to_ref_curr[card_to_ref_curr['CREDITCARDNO'].isin(bad_group1_cards)]['REFERENCE_NUMBER'].unique()

# Filter AECB data for each group
aecb_group0 = aecb_pd_prev[aecb_pd_prev['Reference_Number'].isin(refs_group0)].copy()
aecb_group0['group'] = 0
aecb_group0['group_label'] = 'G0: Bad@2025-06/07 (approved 2024-06→2025-07)'

aecb_group1 = aecb_pd[aecb_pd['Reference_Number'].isin(refs_group1)].copy()
aecb_group1['group'] = 1
aecb_group1['group_label'] = 'G1: Bad@2026-06/07 (approved 2025-06→2026-07)'

print(f"AECB rows — Group 0: {len(aecb_group0):,} | Group 1: {len(aecb_group1):,}")

# --- 2. Merge CC-level features onto AECB for richer comparison ---------------
# Select useful CC columns (non-AECB) to carry over
cc_feature_cols = [
    'REFERENCE_NUMBER', 'CREDITCARDNO', 'APPROVEDATE', 'SNAPSHOT_DATE',
    'DELINQUENCYBUCKET', 'is_bad',
]
# Add any other numeric CC columns that exist
for c in ['CREDITLIMIT', 'OUTSTANDING_BALANCE', 'CURRENTBALANCE',
          'MINIMUM_AMOUNT_DUE', 'SALARY', 'AGE', 'TENURE_MONTHS',
          'APPLICANTSALARY', 'FINALLIMIT']:
    if c in df_thin_prev.columns:
        cc_feature_cols.append(c)
cc_feature_cols = list(dict.fromkeys(cc_feature_cols))  # dedupe, keep order

# Merge CC features for Group 0
cc_g0 = df_thin_prev[df_thin_prev['CREDITCARDNO'].isin(bad_group0_cards)][cc_feature_cols].drop_duplicates('REFERENCE_NUMBER')
aecb_g0_merged = aecb_group0.merge(cc_g0, left_on='Reference_Number', right_on='REFERENCE_NUMBER', how='left')

# Merge CC features for Group 1
cc_feature_cols_curr = [c for c in cc_feature_cols if c in df_thin.columns]
cc_g1 = df_thin[df_thin['CREDITCARDNO'].isin(bad_group1_cards)][cc_feature_cols_curr].drop_duplicates('REFERENCE_NUMBER')
aecb_g1_merged = aecb_group1.merge(cc_g1, left_on='Reference_Number', right_on='REFERENCE_NUMBER', how='left')

# --- 3. Combine into one labelled DataFrame -----------------------------------
bad_combined = pd.concat([aecb_g0_merged, aecb_g1_merged], ignore_index=True)
print(f"\nCombined bad-case dataset: {bad_combined.shape[0]:,} rows × {bad_combined.shape[1]} columns")
print(f"  Group 0 rows: {(bad_combined['group'] == 0).sum():,}")
print(f"  Group 1 rows: {(bad_combined['group'] == 1).sum():,}")

# --- 4. Feature-level comparison (numeric) ------------------------------------
numeric_cols = bad_combined.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ('group', 'is_bad')]

stats_g0 = bad_combined[bad_combined['group'] == 0][numeric_cols].describe().T[['mean', '50%', 'std', 'count']]
stats_g0.columns = ['mean_g0', 'median_g0', 'std_g0', 'count_g0']

stats_g1 = bad_combined[bad_combined['group'] == 1][numeric_cols].describe().T[['mean', '50%', 'std', 'count']]
stats_g1.columns = ['mean_g1', 'median_g1', 'std_g1', 'count_g1']

comparison = stats_g0.join(stats_g1)
comparison['mean_diff'] = comparison['mean_g1'] - comparison['mean_g0']
comparison['mean_pct_change'] = (comparison['mean_diff'] / comparison['mean_g0'].replace(0, np.nan)) * 100
comparison['median_diff'] = comparison['median_g1'] - comparison['median_g0']
comparison = comparison.sort_values('mean_pct_change', ascending=False, key=abs)

print("\n" + "=" * 90)
print("Top 30 features with largest mean % change  (Group 0 → Group 1)")
print("=" * 90)
display(comparison.head(30))

# --- 5. Visual: top 20 features by absolute mean % change ---------------------
top20 = comparison.head(20).copy()
top20 = top20.sort_values('mean_pct_change')

fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [3, 2]})

# Left: horizontal bar of mean % change
ax = axes[0]
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in top20['mean_pct_change']]
ax.barh(top20.index, top20['mean_pct_change'], color=colors)
ax.set_xlabel('Mean % Change (G0 → G1)', fontsize=11)
ax.set_title('Top 20 Feature Changes (mean % change)\n'
             'G0: Bad@2025-06/07 approved 2024-06→2025-07  |  '
             'G1: Bad@2026-06/07 approved 2025-06→2026-07', fontsize=11)
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')

# Right: side-by-side mean values for context
ax2 = axes[1]
y_pos = range(len(top20))
bar_h = 0.35
ax2.barh([y - bar_h/2 for y in y_pos], top20['mean_g0'], height=bar_h,
         color='#3498db', label='G0 mean', alpha=0.8)
ax2.barh([y + bar_h/2 for y in y_pos], top20['mean_g1'], height=bar_h,
         color='#e67e22', label='G1 mean', alpha=0.8)
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(top20.index, fontsize=8)
ax2.set_xlabel('Mean Value', fontsize=11)
ax2.set_title('Side-by-Side Mean Values', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# --- 6. Print the combined labelled dataset summary ---------------------------
print("\n" + "=" * 90)
print("Labelled bad-case dataset stored in  `bad_combined`  (group=0 early, group=1 latest)")
print("=" * 90)
print(f"Columns: {bad_combined.columns.tolist()[:15]} ... ({bad_combined.shape[1]} total)")
print(f"\nSample rows:")
display(bad_combined.head(5))

In [ ]:
# ============================================================================
# Simple Decision Tree: Can we separate G0 Bad from G1 Bad using AECB + CC features?
# ============================================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- 1. Prepare features & label ---------------------------------------------
# Target: group column (0 = G0, 1 = G1)
df_model = pd.concat([aecb_g0_merged, aecb_g1_merged], ignore_index=True).copy()

# Keep only numeric columns, drop identifiers and target
drop_cols = ['group', 'group_label', 'is_bad', 'CREDITCARDNO', 'REFERENCE_NUMBER',
             'Reference_Number', 'APPROVEDATE', 'SNAPSHOT_DATE']
feature_cols = [
    c for c in df_model.select_dtypes(include='number').columns
    if c not in drop_cols
]

X = df_model[feature_cols].copy()
y = df_model['group'].astype(int)

# Fill NaN with -999 so the tree can split on missingness
X = X.fillna(-999)

print(f"Features: {len(feature_cols)} | Samples: {len(X):,} (G0={int((y==0).sum()):,}, G1={int((y==1).sum()):,})")

# --- 2. Train / Test split ----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# --- 3. Fit a single shallow Decision Tree (regularised to reduce overfitting) -
# Shallow depth (max_depth=3) + large minimum leaf size (min_samples_leaf=200)
# → sacrifices some training accuracy for better generalisation on test set.
dt = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=80,
    min_samples_split=80,
    random_state=42,
)
dt.fit(X_train, y_train)

train_auc = roc_auc_score(y_train, dt.predict_proba(X_train)[:, 1])
print(f"\n✅ Decision Tree (max_depth=3, min_samples_leaf=200)")
print(f"   Tree depth : {dt.get_depth()} | Leaves: {dt.get_n_leaves()}")
print(f"   Train AUC  : {train_auc:.4f}")

# --- 4. Evaluate on TEST set --------------------------------------------------
y_pred = dt.predict(X_test)
y_prob = dt.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"\nTest AUC: {auc:.4f}")
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred, target_names=['G0 Bad', 'G1 Bad']))

# --- 5. Confusion Matrix ------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['G0 Bad', 'G1 Bad'], ax=axes[0], cmap='Blues'
)
axes[0].set_title(f'Confusion Matrix (Test)  —  AUC={auc:.4f}', fontsize=12)

# --- 6. Feature Importance (top 15) -------------------------------------------
if hasattr(dt, 'feature_importances_'):
    importances = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
elif hasattr(dt, 'coef_'):
    importances = pd.Series(np.abs(dt.coef_[0]), index=feature_cols).sort_values(ascending=False)
else:
    importances = None

if importances is not None:
    top15_imp = importances.head(15).sort_values()
    axes[1].barh(top15_imp.index, top15_imp.values, color='#3498db', edgecolor='white')
    imp_label = 'Feature Importance (Gini)' if hasattr(dt, 'feature_importances_') else 'Absolute Coefficient'
    axes[1].set_xlabel(imp_label, fontsize=11)
    axes[1].set_title('Top 15 Features Separating G0 Bad vs G1 Bad', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='x')
else:
    axes[1].text(0.5, 0.5, 'Feature importance not available', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

# --- 7. Interpretation: Can G0 Bad be separated from G1 Bad? -----------------
print("\n" + "=" * 80)
print("INTERPRETATION: Can G0 Bad be separated from G1 Bad?")
print("=" * 80)

from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print(f"\n  Test AUC       : {auc:.4f}")
print(f"  Precision (G1) : {precision:.4f}")
print(f"  Recall    (G1) : {recall:.4f}")
print(f"  F1 Score  (G1) : {f1:.4f}")

# Train-set AUC for overfitting check
y_prob_train = dt.predict_proba(X_train)[:, 1]
auc_train = roc_auc_score(y_train, y_prob_train)
print(f"  Train AUC      : {auc_train:.4f}")
print(f"  AUC Gap (train-test): {auc_train - auc:.4f}")

# Verdict
if auc >= 0.75:
    verdict = (
        "YES — the two groups are meaningfully separable.\n"
        f"  A shallow depth-5 tree achieves AUC={auc:.4f}, which indicates that the\n"
        "  AECB + CC features carry substantial discriminative signal between\n"
        "  G0 Bad (earlier vintage) and G1 Bad (later vintage).\n"
        "  The top features in the importance chart highlight what changed most\n"
        "  between the two cohorts."
    )
elif auc >= 0.60:
    verdict = (
        "PARTIALLY — there is some separability, but it is modest.\n"
        f"  AUC={auc:.4f} is above chance (0.50) but below the 0.75 threshold for\n"
        "  strong separation. The groups overlap considerably, meaning the \n"
        "  bad-card profiles shifted only moderately between vintages."
    )
else:
    verdict = (
        "NO — the two groups are NOT meaningfully separable.\n"
        f"  AUC={auc:.4f} is close to random (0.50), suggesting that G0 Bad and\n"
        "  G1 Bad have very similar AECB + CC feature distributions.\n"
        "  The bad-card profiles did not change much between vintages."
    )

if (auc_train - auc) > 0.10:
    verdict += (
        f"\n\n  ⚠️  Overfitting warning: train AUC ({auc_train:.4f}) is notably higher\n"
        f"  than test AUC ({auc:.4f}). The true separability may be weaker than\n"
        "  the train metrics suggest."
    )

print(f"\n  VERDICT: {verdict}")
print("=" * 80)

# --- 8. Print the tree splits (text) -----------------------------------------
from sklearn.tree import export_text
print("\nDecision Tree Structure (max_depth=5):")
print(export_text(dt, feature_names=feature_cols, max_depth=5))

In [ ]:
def compute_psi_v2(expected, actual, bins=10, eps=1e-4,
                   extreme_thresholds=(99, 999, 9999, -1, -99)):
    """
    Compute PSI with special handling for extreme / default-fill values.

    Values whose absolute value matches any entry in `extreme_thresholds`
    are bucketed separately (one bucket per distinct extreme value).
    The remaining ("normal") values are quantile-binned as usual.

    Returns
    -------
    dict with keys:
        psi_full      – PSI computed over ALL buckets (extreme + normal)
        psi_no_extreme – PSI computed only on the normal-value buckets
        n_extreme_exp  – # extreme rows in expected
        n_extreme_act  – # extreme rows in actual
        detail         – per-bucket DataFrame (for debugging / display)
    """
    import pandas as pd

    exp = pd.to_numeric(pd.Series(expected), errors='coerce').dropna().values
    act = pd.to_numeric(pd.Series(actual),   errors='coerce').values  # keep NaN for mask
    act = act[~np.isnan(act)]

    if len(exp) < bins or len(act) < bins:
        return {'psi_full': np.nan, 'psi_no_extreme': np.nan,
                'n_extreme_exp': 0, 'n_extreme_act': 0, 'detail': None}

    thresholds = set(extreme_thresholds)

    # --- split extreme vs normal ------------------------------------------------
    exp_extreme_mask = np.isin(exp, list(thresholds))
    act_extreme_mask = np.isin(act, list(thresholds))

    exp_normal = exp[~exp_extreme_mask]
    act_normal = act[~act_extreme_mask]
    exp_extreme = exp[exp_extreme_mask]
    act_extreme = act[act_extreme_mask]

    # --- buckets for extreme values (one bucket per distinct extreme value) ------
    all_extreme_vals = sorted(set(np.concatenate([exp_extreme, act_extreme]))) if \
        (len(exp_extreme) + len(act_extreme)) > 0 else []

    bucket_rows = []  # list of (label, exp_count, act_count)
    for v in all_extreme_vals:
        bucket_rows.append((
            f'extreme={v}',
            np.sum(exp_extreme == v),
            np.sum(act_extreme == v),
        ))

    # --- quantile-bin the normal values -----------------------------------------
    if len(exp_normal) >= bins:
        breakpoints = np.unique(np.percentile(exp_normal, np.linspace(0, 100, bins + 1)))
    else:
        breakpoints = np.array([])

    if len(breakpoints) > 1:
        exp_hist = np.histogram(exp_normal, bins=breakpoints)[0]
        act_hist = np.histogram(act_normal, bins=breakpoints)[0]
        for i in range(len(exp_hist)):
            lo, hi = breakpoints[i], breakpoints[i + 1]
            bucket_rows.append((f'({lo:.4g}, {hi:.4g}]', exp_hist[i], act_hist[i]))
    elif len(exp_normal) > 0:
        # fallback: single bucket for all normal values
        bucket_rows.append(('normal_all', len(exp_normal), len(act_normal)))

    # --- build detail DataFrame & compute PSI -----------------------------------
    detail = pd.DataFrame(bucket_rows, columns=['bucket', 'exp_count', 'act_count'])
    detail['exp_count'] = detail['exp_count'] + eps
    detail['act_count'] = detail['act_count'] + eps

    # Full PSI (all buckets)
    exp_total = detail['exp_count'].sum()
    act_total = detail['act_count'].sum()
    detail['exp_pct'] = detail['exp_count'] / exp_total
    detail['act_pct'] = detail['act_count'] / act_total
    detail['psi_bucket'] = (detail['act_pct'] - detail['exp_pct']) * \
                            np.log(detail['act_pct'] / detail['exp_pct'])
    psi_full = detail['psi_bucket'].sum()

    # PSI without extreme buckets
    normal_mask = ~detail['bucket'].str.startswith('extreme=')
    normal_detail = detail[normal_mask]
    if len(normal_detail) > 0:
        exp_total_n = normal_detail['exp_count'].sum()
        act_total_n = normal_detail['act_count'].sum()
        exp_pct_n = normal_detail['exp_count'] / exp_total_n
        act_pct_n = normal_detail['act_count'] / act_total_n
        psi_no_extreme = float(np.sum((act_pct_n - exp_pct_n) * np.log(act_pct_n / exp_pct_n)))
    else:
        psi_no_extreme = np.nan

    return {
        'psi_full': float(psi_full),
        'psi_no_extreme': psi_no_extreme,
        'n_extreme_exp': int(exp_extreme_mask.sum()),
        'n_extreme_act': int(act_extreme_mask.sum()),
        'detail': detail,
    }

In [ ]:
# ============================================================================
# PSI (Population Stability Index) – Top 15 features with biggest shift
# from AECB Group 0 → Group 1
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

# --- Identify numeric columns common to both groups --------------------------
numeric_cols_g0 = set(aecb_group0.select_dtypes(include='number').columns)
numeric_cols_g1 = set(aecb_group1.select_dtypes(include='number').columns)
common_numeric = sorted(numeric_cols_g0 & numeric_cols_g1 - {'group'})

# --- Compute PSI for every common numeric feature ----------------------------
psi_results = []
for col in common_numeric:
    result = compute_psi_v2(aecb_group0[col], aecb_group1[col], bins=10)
    psi_results.append({'feature': col, 'PSI': result['psi_full'], 'PSI_no_extreme': result['psi_no_extreme']})

psi_df = (
    pd.DataFrame(psi_results)
    .dropna(subset=['PSI'])
    .sort_values('PSI', ascending=False)
    .reset_index(drop=True)
)

# --- Display top 15 ----------------------------------------------------------
top15 = psi_df.head(15).copy()
print("Top 15 AECB Features by PSI  (Group 0 → Group 1)")
print("="*60)
print("PSI < 0.10 → No significant shift")
print("PSI 0.10–0.25 → Moderate shift")
print("PSI > 0.25 → Major shift")
print("="*60)
display(top15)

# --- Horizontal bar chart ----------------------------------------------------
top15_plot = top15.sort_values('PSI')
colors = ['#e74c3c' if v > 0.25 else '#f39c12' if v > 0.10 else '#2ecc71'
          for v in top15_plot['PSI']]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top15_plot['feature'], top15_plot['PSI'], color=colors, edgecolor='white')
ax.axvline(0.10, color='orange', linestyle='--', linewidth=1, label='Moderate (0.10)')
ax.axvline(0.25, color='red',    linestyle='--', linewidth=1, label='Major (0.25)')
ax.set_xlabel('PSI', fontsize=12)
ax.set_title('Top 15 AECB Features by Population Stability Index\n'
             'G0: Bad@2025-06/07 (approved 2024-06→2025-07)  →  '
             'G1: Bad@2026-06/07 (approved 2025-06→2026-07)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Histogram + KDE density plot: pl_mob_recent — G0 Bad vs G1 Bad
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np

feat = 'MOB_ACTIVE'

g0_data = aecb_group0[feat].dropna()
g1_data = aecb_group1[feat].dropna()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Left: Overlapping histograms -------------------------------------------
bin_lo = min(g0_data.min(), g1_data.min())
bin_hi = max(g0_data.max(), g1_data.max())
bins = np.linspace(bin_lo, bin_hi, 40)

axes[0].hist(g0_data, bins=bins, alpha=0.55, color='#3498db', edgecolor='white',
             label=f'G0 Bad (n={len(g0_data):,})', density=True)
axes[0].hist(g1_data, bins=bins, alpha=0.55, color='#e74c3c', edgecolor='white',
             label=f'G1 Bad (n={len(g1_data):,})', density=True)
axes[0].set_xlabel(feat, fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title(f'Histogram (density) — {feat}', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Right: KDE density curves -----------------------------------------------
from scipy.stats import gaussian_kde

x_grid = np.linspace(bin_lo, bin_hi, 500)

kde_g0 = gaussian_kde(g0_data)
kde_g1 = gaussian_kde(g1_data)

axes[1].plot(x_grid, kde_g0(x_grid), color='#3498db', linewidth=2,
             label=f'G0 Bad (n={len(g0_data):,})')
axes[1].fill_between(x_grid, kde_g0(x_grid), alpha=0.25, color='#3498db')

axes[1].plot(x_grid, kde_g1(x_grid), color='#e74c3c', linewidth=2,
             label=f'G1 Bad (n={len(g1_data):,})')
axes[1].fill_between(x_grid, kde_g1(x_grid), alpha=0.25, color='#e74c3c')

axes[1].set_xlabel(feat, fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title(f'KDE Density — {feat}', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# --- Summary stats -----------------------------------------------------------
fig.suptitle(
    f'{feat}:  G0 mean={g0_data.mean():.2f}, median={g0_data.median():.2f}  |  '
    f'G1 mean={g1_data.mean():.2f}, median={g1_data.median():.2f}',
    fontsize=11, y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Define "Good" cards using a rolling 4-month window, evaluated ONLY at the
# same observation snapshot months used for Bad (cell 13).
# For snapshot M the window is [M-3, M-2, M-1, M]:
#   - All 4 months present AND max(DELINQUENCYBUCKET) == 0 → Good
#   - Fewer than 4 months available                        → Unknown
#   - Otherwise                                            → Not Good
# ============================================================================
import numpy as np

def get_good_cards_rolling4m(df, eval_months):
    """
    Rolling 4-calendar-month window evaluated ONLY at the specified
    observation snapshot months (same months used for Bad definition).

    Parameters
    ----------
    df          : DataFrame with CREDITCARDNO, SNAPSHOT_DATE, DELINQUENCYBUCKET
    eval_months : list of snapshot month strings, e.g. ['2026-06', '2026-07']

    Returns
    -------
    good_cards : array of qualifying CREDITCARDNOs
    """
    sub = df.copy()
    sub['snapshot_month'] = pd.to_datetime(sub['SNAPSHOT_DATE']).dt.to_period('M')
    sub['DELINQUENCYBUCKET'] = pd.to_numeric(sub['DELINQUENCYBUCKET'], errors='coerce')

    # One row per card × snapshot_month: max delinquency in that month
    card_snap = (
        sub.groupby(['CREDITCARDNO', 'snapshot_month'], as_index=False)
        .agg(max_bucket=('DELINQUENCYBUCKET', 'max'))
    )

    # Only build evaluation targets for the specified observation months
    eval_periods = [pd.Period(m, 'M') for m in eval_months]
    card_snap_eval = card_snap[card_snap['snapshot_month'].isin(eval_periods)]
    print(f"  Cards with data at observation months {eval_months}: {card_snap_eval['CREDITCARDNO'].nunique():,}")

    # For each card × eval snapshot, build 4 target calendar months [M-3…M]
    target_frames = []
    for offset in range(4):
        tmp = card_snap_eval[['CREDITCARDNO', 'snapshot_month']].copy()
        tmp['eval_month'] = tmp['snapshot_month']
        tmp['target_month'] = tmp['snapshot_month'] - offset
        target_frames.append(tmp[['CREDITCARDNO', 'eval_month', 'target_month']])
    targets = pd.concat(target_frames, ignore_index=True)

    # Left-join to get actual bucket at each target month (from ALL months)
    targets = targets.merge(
        card_snap.rename(columns={'snapshot_month': 'target_month'}),
        on=['CREDITCARDNO', 'target_month'],
        how='left'
    )

    # Aggregate per card × eval_month
    eval_stats = (
        targets.groupby(['CREDITCARDNO', 'eval_month'])
        .agg(n_months=('max_bucket', 'count'), max_bucket_4m=('max_bucket', 'max'))
        .reset_index()
    )

    eval_stats['status'] = np.where(
        eval_stats['n_months'] < 4, 'Unknown',
        np.where(eval_stats['max_bucket_4m'] == 0, 'Good', 'Not Good')
    )

    # Status distribution at observation months
    status_dist = eval_stats['status'].value_counts()
    print(f"  Status at {eval_months}:")
    for s, c in status_dist.items():
        print(f"    {s}: {c:,}")

    # Final label: latest eval_month per card among observation months
    latest_eval = (
        eval_stats.sort_values('eval_month')
        .groupby('CREDITCARDNO').last()
        .reset_index()
    )

    good_cards  = latest_eval[latest_eval['status'] == 'Good']['CREDITCARDNO'].unique()
    n_unknown   = (latest_eval['status'] == 'Unknown').sum()
    n_not_good  = (latest_eval['status'] == 'Not Good').sum()

    print(f"  Final — Good: {len(good_cards):,} | Unknown: {n_unknown:,} | Not Good: {n_not_good:,}")

    return good_cards


# --- Group 0: Good at same observation months as Bad (2025-06/07) -------------
print("Group 0 (approved 2024-06→2025-07, observed @ 2025-06/07):")
good_g0_cards = get_good_cards_rolling4m(df_thin_prev, eval_months=['2025-06', '2025-07'])

# --- Group 1: Good at same observation months as Bad (2026-06/07) -------------
print("\nGroup 1 (approved 2025-06→2026-07, observed @ 2026-06/07):")
good_g1_cards = get_good_cards_rolling4m(df_thin, eval_months=['2026-06', '2026-07'])

# --- Map to REFERENCE_NUMBER and pull AECB data --------------------------------
# Group 0
card_to_ref_prev = df_thin_prev[['CREDITCARDNO', 'REFERENCE_NUMBER']].drop_duplicates()
refs_good_g0 = card_to_ref_prev[card_to_ref_prev['CREDITCARDNO'].isin(good_g0_cards)]['REFERENCE_NUMBER'].unique()

aecb_clean_g0 = aecb_pd_prev[aecb_pd_prev['Reference_Number'].isin(refs_good_g0)].copy()
aecb_clean_g0['group'] = 0
aecb_clean_g0['group_label'] = 'G0: Good @ 2025-06/07 (approved 2024-06→2025-07)'

# Group 1
card_to_ref_curr = df_thin[['CREDITCARDNO', 'REFERENCE_NUMBER']].drop_duplicates()
refs_good_g1 = card_to_ref_curr[card_to_ref_curr['CREDITCARDNO'].isin(good_g1_cards)]['REFERENCE_NUMBER'].unique()

aecb_clean_g1 = aecb_pd[aecb_pd['Reference_Number'].isin(refs_good_g1)].copy()
aecb_clean_g1['group'] = 1
aecb_clean_g1['group_label'] = 'G1: Good @ 2026-06/07 (approved 2025-06→2026-07)'

# --- Summary -------------------------------------------------------------------
print("\n" + "=" * 80)
print("AECB data for Good cards (rolling 4m window at same observation months as Bad)")
print("=" * 80)
print(f"  aecb_clean_g0: {len(aecb_clean_g0):,} rows  ({len(refs_good_g0):,} unique refs)")
print(f"  aecb_clean_g1: {len(aecb_clean_g1):,} rows  ({len(refs_good_g1):,} unique refs)")


In [ ]:
aecb_group1.head(2)

In [ ]:
# ============================================================================
# PSI: aecb_group1 (all bad G1) vs aecb_clean_g1 (rolling-4m Good G1)
# Top 15 AECB features with largest Population Stability Index
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt



# --- Identify common numeric columns -----------------------------------------
numeric_cols_all  = set(aecb_group1.select_dtypes(include='number').columns)
numeric_cols_clean = set(aecb_clean_g1.select_dtypes(include='number').columns)
common_numeric = sorted(numeric_cols_all & numeric_cols_clean - {'group'})

# --- Compute PSI (full & no-extreme) for every common numeric feature --------
psi_results = []
for col in common_numeric:
    res = compute_psi_v2(aecb_group1[col], aecb_clean_g1[col], bins=10,
                         extreme_thresholds=(99, 999, 9999, -1, -99))
    psi_results.append({
        'feature': col,
        'PSI_full': res['psi_full'],
        'PSI_no_extreme': res['psi_no_extreme'],
        'n_extreme_exp': res['n_extreme_exp'],
        'n_extreme_act': res['n_extreme_act'],
    })

psi_df_g1 = (
    pd.DataFrame(psi_results)
    .dropna(subset=['PSI_full'])
    .sort_values('PSI_full', ascending=False)
    .reset_index(drop=True)
)

# --- Display top 15 -----------------------------------------------------------
top15 = psi_df_g1.head(15).copy()
print("Top 15 AECB Features by PSI  (aecb_group1 → aecb_clean_g1)")
print("=" * 60)
print("PSI < 0.10 → No significant shift")
print("PSI 0.10–0.25 → Moderate shift")
print("PSI > 0.25 → Major shift")
print("=" * 60)
print("PSI_full = including extreme/default buckets")
print("PSI_no_extreme = excluding extreme/default buckets")
print("=" * 60)
display(top15)

# --- Horizontal bar chart: side-by-side Full vs No-Extreme --------------------
top15_plot = top15.sort_values('PSI_full')

fig, axes = plt.subplots(1, 2, figsize=(22, 8), sharey=True)

# Left: PSI_full
colors_full = ['#e74c3c' if v > 0.25 else '#f39c12' if v > 0.10 else '#2ecc71'
               for v in top15_plot['PSI_full']]
axes[0].barh(top15_plot['feature'], top15_plot['PSI_full'],
             color=colors_full, edgecolor='white')
axes[0].axvline(0.10, color='orange', linestyle='--', linewidth=1, label='Moderate (0.10)')
axes[0].axvline(0.25, color='red',    linestyle='--', linewidth=1, label='Major (0.25)')
axes[0].set_xlabel('PSI (Full, incl. extreme buckets)', fontsize=12)
axes[0].set_title('PSI Full (with extreme/default values)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='x')

# Right: PSI_no_extreme
colors_no = ['#e74c3c' if v > 0.25 else '#f39c12' if v > 0.10 else '#2ecc71'
             for v in top15_plot['PSI_no_extreme']]
axes[1].barh(top15_plot['feature'], top15_plot['PSI_no_extreme'],
             color=colors_no, edgecolor='white')
axes[1].axvline(0.10, color='orange', linestyle='--', linewidth=1, label='Moderate (0.10)')
axes[1].axvline(0.25, color='red',    linestyle='--', linewidth=1, label='Major (0.25)')
axes[1].set_xlabel('PSI (No Extreme)', fontsize=12)
axes[1].set_title('PSI Without Extreme/Default Values', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='x')

fig.suptitle('Top 15 AECB Features by PSI\n'
             'Expected: aecb_group1 (all bad @ 2026-06/07)  →  '
             'Actual: aecb_clean_g1 (rolling-4m clean, Good G1)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Histogram + KDE density plot: MOB_Since_Chq_Rtn — Bad G1 vs Good G1
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

feat = 'L3M_NRAK_NOF_ENQURY'

g1_bad_data  = pd.to_numeric(aecb_group1[feat], errors='coerce').dropna()
g1_good_data = pd.to_numeric(aecb_clean_g1[feat], errors='coerce').dropna()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Left: Overlapping histograms --------------------------------------------
bin_lo = min(g1_bad_data.min(), g1_good_data.min())
bin_hi = max(g1_bad_data.max(), g1_good_data.max())
bins = np.linspace(bin_lo, bin_hi, 40)

axes[0].hist(g1_bad_data, bins=bins, alpha=0.55, color='#e74c3c', edgecolor='white',
             label=f'G1 Bad (n={len(g1_bad_data):,})', density=True)
axes[0].hist(g1_good_data, bins=bins, alpha=0.55, color='#2ecc71', edgecolor='white',
             label=f'G1 Good (n={len(g1_good_data):,})', density=True)
axes[0].set_xlabel(feat, fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title(f'Histogram (density) — {feat}', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Right: KDE density curves ------------------------------------------------
x_grid = np.linspace(bin_lo, bin_hi, 500)

kde_bad  = gaussian_kde(g1_bad_data)
kde_good = gaussian_kde(g1_good_data)

axes[1].plot(x_grid, kde_bad(x_grid), color='#e74c3c', linewidth=2,
             label=f'G1 Bad (n={len(g1_bad_data):,})')
axes[1].fill_between(x_grid, kde_bad(x_grid), alpha=0.25, color='#e74c3c')

axes[1].plot(x_grid, kde_good(x_grid), color='#2ecc71', linewidth=2,
             label=f'G1 Good (n={len(g1_good_data):,})')
axes[1].fill_between(x_grid, kde_good(x_grid), alpha=0.25, color='#2ecc71')

axes[1].set_xlabel(feat, fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title(f'KDE Density — {feat}', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# --- Summary stats ------------------------------------------------------------
fig.suptitle(
    f'{feat}:  Bad mean={g1_bad_data.mean():.2f}, median={g1_bad_data.median():.2f}  |  '
    f'Good mean={g1_good_data.mean():.2f}, median={g1_good_data.median():.2f}',
    fontsize=11, y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Combine aecb_group1 (bad) & aecb_clean_g1 (good), bucket MOB_Since_Chq_Rtn
# ============================================================================
import numpy as np
import pandas as pd

# Label: aecb_group1 = bad (1), aecb_clean_g1 = good (0)
bad_df = aecb_group1.copy()
bad_df['target'] = 1
bad_df['target_label'] = 'Bad'

good_df = aecb_clean_g1.copy()
good_df['target'] = 0
good_df['target_label'] = 'Good'

# Combine
combined = pd.concat([bad_df, good_df], ignore_index=True)

# Fill missing MOB_Since_Chq_Rtn with 99 (aligned with Cell 16 imputation)
combined['MOB_Since_Chq_Rtn'] = pd.to_numeric(combined['MOB_Since_Chq_Rtn'], errors='coerce').fillna(99)

print(f"Combined rows: {len(combined):,}  (Bad: {(combined['target']==1).sum():,} | Good: {(combined['target']==0).sum():,})")
print(f"MOB_Since_Chq_Rtn missing filled with 99: {(combined['MOB_Since_Chq_Rtn'] == 99).sum():,}")

# --- Aggregate: good vs bad count per distinct MOB_Since_Chq_Rtn value --------
value_stats = (
    combined
    .groupby('MOB_Since_Chq_Rtn')
    .agg(
        total=('target', 'count'),
        bad=('target', 'sum'),
    )
    .reset_index()
)
value_stats['good'] = value_stats['total'] - value_stats['bad']
value_stats['bad_rate'] = (value_stats['bad'] / value_stats['total'] * 100).round(2)
value_stats = value_stats[['MOB_Since_Chq_Rtn', 'total', 'good', 'bad', 'bad_rate']].sort_values('MOB_Since_Chq_Rtn')

print("\n" + "=" * 70)
print("MOB_Since_Chq_Rtn — Good vs Bad per Value")
print("=" * 70)
display(value_stats)

# --- Plot: stacked bar --------------------------------------------------------
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(14, 6))
x_labels = value_stats['MOB_Since_Chq_Rtn'].astype(int).astype(str)
x_pos = range(len(x_labels))

ax1.bar(x_pos, value_stats['good'].values, label='Good', color='#2ecc71', edgecolor='white')
ax1.bar(x_pos, value_stats['bad'].values, bottom=value_stats['good'].values, label='Bad', color='#e74c3c', edgecolor='white')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax1.set_xlabel('MOB_Since_Chq_Rtn', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('MOB_Since_Chq_Rtn — Good vs Bad per Value\n(aecb_group1 [Bad] + aecb_clean_g1 [Good], missing → 99)', fontsize=13)
ax1.legend(fontsize=11)

# Overlay bad rate line
ax2 = ax1.twinx()
ax2.plot(x_pos, value_stats['bad_rate'].values, color='navy', marker='o', linewidth=2, label='Bad Rate %')
ax2.set_ylabel('Bad Rate %', fontsize=12, color='navy')
ax2.tick_params(axis='y', labelcolor='navy')
ax2.legend(loc='upper left', fontsize=10)

ax1.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Bin PL_MOB_recent into 10 bins and show Good vs Bad per bin
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt

combined['L3M_NRAK_NOF_ENQURY'] = pd.to_numeric(combined['L3M_NRAK_NOF_ENQURY'], errors='coerce')

# Cut into 10 bins (equal-width)
combined['L3M_NRAK_NOF_ENQURY_bin'] = pd.cut(combined['L3M_NRAK_NOF_ENQURY'], bins=10)

# Aggregate good / bad counts per bin
bin_stats = (
    combined
    .groupby('L3M_NRAK_NOF_ENQURY_bin', observed=False)
    .agg(
        total=('target', 'count'),
        bad=('target', 'sum'),
    )
    .reset_index()
)
bin_stats['good'] = bin_stats['total'] - bin_stats['bad']
bin_stats['bad_rate'] = (bin_stats['bad'] / bin_stats['total'] * 100).round(2)
bin_stats = bin_stats[['L3M_NRAK_NOF_ENQURY_bin', 'total', 'good', 'bad', 'bad_rate']]

print("L3M_NRAK_NOF_ENQURY — Good vs Bad per Bin (10 equal-width bins)")
print("=" * 70)
display(bin_stats)

# --- Plot: stacked bar + bad rate line ----------------------------------------
fig, ax1 = plt.subplots(figsize=(14, 6))
x_labels = bin_stats['L3M_NRAK_NOF_ENQURY_bin'].astype(str)
x_pos = range(len(x_labels))

ax1.bar(x_pos, bin_stats['good'].values, label='Good', color='#2ecc71', edgecolor='white')
ax1.bar(x_pos, bin_stats['bad'].values, bottom=bin_stats['good'].values, label='Bad', color='#e74c3c', edgecolor='white')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax1.set_xlabel('L3M_NRAK_NOF_ENQURY Bin', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('L3M_NRAK_NOF_ENQURY — Good vs Bad per Bin (10 bins)', fontsize=13)
ax1.legend(fontsize=11)

ax2 = ax1.twinx()
ax2.plot(x_pos, bin_stats['bad_rate'].values, color='navy', marker='o', linewidth=2, label='Bad Rate %')
ax2.set_ylabel('Bad Rate %', fontsize=12, color='navy')
ax2.tick_params(axis='y', labelcolor='navy')
ax2.legend(loc='upper left', fontsize=10)

ax1.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Simple Decision Tree: find the segment with the highest bad rate to knock out
# Uses combined (aecb_group1 bad + aecb_clean_g1 good) from earlier cells
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# --- 1. Prepare features & target --------------------------------------------
# Select numeric AECB columns (drop identifiers & meta columns)
exclude_cols = {'group', 'group_label', 'target', 'target_label',
                'Reference_Number', 'REFERENCE_NUMBER', 'CREDITCARDNO',
                'APPROVEDATE', 'SNAPSHOT_DATE', 'is_bad', 'DELINQUENCYBUCKET'}

feature_cols = [
    c for c in combined.select_dtypes(include='number').columns
    if c not in exclude_cols
]

X = combined[feature_cols].copy()
y = combined['target'].copy()

# Fill missing values with -1 so the tree can split on "missing vs present"
X = X.fillna(-1)

print(f"Features: {len(feature_cols)} | Rows: {len(X):,}")
print(f"Bad (target=1): {y.sum():,}  |  Good (target=0): {(y==0).sum():,}")
print(f"Overall bad rate: {y.mean()*100:.2f}%")

# --- 2. Train a shallow decision tree (easy to interpret) ---------------------
tree = DecisionTreeClassifier(
    max_depth=3,          # shallow → interpretable
    min_samples_leaf=200,  # avoid tiny noisy leaves
    random_state=42       # no class_weight='balanced' so tree values = actual counts
)
tree.fit(X, y)

# --- 3. Visualise the tree ----------------------------------------------------
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    tree,
    feature_names=feature_cols,
    class_names=['Good', 'Bad'],
    filled=True,
    rounded=True,
    fontsize=9,
    proportion=False,   # show actual sample counts, not weighted proportions
    ax=ax
)
ax.set_title('Decision Tree — Find the Highest Bad-Rate Segment to Knock Out First',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# --- 4. Print text rules for easy reading -------------------------------------
print("\n" + "=" * 80)
print("TREE RULES (text)")
print("=" * 80)
print(export_text(tree, feature_names=feature_cols, decimals=2))

# --- 5. Leaf-node summary: bad rate & volume per terminal node ----------------
leaf_ids = tree.apply(X)
leaf_summary = (
    pd.DataFrame({'leaf': leaf_ids, 'target': y.values})
    .groupby('leaf')
    .agg(total=('target', 'count'), bads=('target', 'sum'))
    .reset_index()
)
leaf_summary['goods'] = leaf_summary['total'] - leaf_summary['bads']
leaf_summary['bad_rate_pct'] = (leaf_summary['bads'] / leaf_summary['total'] * 100).round(2)
leaf_summary['pop_pct'] = (leaf_summary['total'] / leaf_summary['total'].sum() * 100).round(2)
leaf_summary = leaf_summary.sort_values('bad_rate_pct', ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("LEAF NODE SUMMARY  (sorted by bad rate — top = best knock-out candidate)")
print("=" * 80)
display(leaf_summary)

# --- 6. Identify the worst leaf & print actionable knock-out rule -------------
worst = leaf_summary.iloc[0]
worst_leaf_id = int(worst['leaf'])

# Walk the tree to extract the rule path for this leaf
from sklearn.tree import _tree

def get_leaf_rules(tree_model, feature_names, leaf_id):
    """Return the conjunction of split rules leading to a given leaf node."""
    tree_ = tree_model.tree_
    rules = []

    def recurse(node, path):
        if tree_.children_left[node] == _tree.TREE_LEAF:
            if node == leaf_id:
                rules.extend(path)
            return
        feat = feature_names[tree_.feature[node]]
        thresh = tree_.threshold[node]
        recurse(tree_.children_left[node],  path + [f"{feat} <= {thresh:.2f}"])
        recurse(tree_.children_right[node], path + [f"{feat} > {thresh:.2f}"])

    recurse(0, [])
    return rules

knockout_rules = get_leaf_rules(tree, feature_cols, worst_leaf_id)

print("\n" + "=" * 80)
print("🎯 RECOMMENDED KNOCK-OUT SEGMENT  (highest bad rate leaf)")
print("=" * 80)
print(f"  Bad Rate : {worst['bad_rate_pct']:.2f}%")
print(f"  Volume   : {int(worst['total']):,} ({worst['pop_pct']:.1f}% of population)")
print(f"  Bads     : {int(worst['bads']):,}")
print(f"  Rule     :")
for r in knockout_rules:
    print(f"    AND  {r}")
print("\nApplying this rule as a decline policy would remove the segment above.")

# --- 7. Feature importance bar chart ------------------------------------------
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importances['feature'][::-1], importances['importance'][::-1],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Gini Importance', fontsize=12)
ax.set_title('Top 10 Features — Decision Tree Split Importance', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
feature_cols
X.columns

In [ ]:
# ============================================================================
# Filter df_thin where L3M_NRAK_PL_NOF_ENQURY > 0.5
# Check DELINQUENCYBUCKET distribution per snapshot_month
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt

# Ensure numeric types
df_thin['L3M_NRAK_PL_NOF_ENQURY'] = pd.to_numeric(df_thin['L3M_NRAK_PL_NOF_ENQURY'], errors='coerce')
df_thin['DELINQUENCYBUCKET'] = pd.to_numeric(df_thin['DELINQUENCYBUCKET'], errors='coerce')
df_thin['snapshot_month'] = pd.to_datetime(df_thin['SNAPSHOT_DATE']).dt.to_period('M')

# Filter: L3M_NRAK_PL_NOF_ENQURY > 0.5 AND L3M_NRAK_NOF_ENQURY > 5.5
df_thin['L3M_NRAK_NOF_ENQURY'] = pd.to_numeric(df_thin['L3M_NRAK_NOF_ENQURY'], errors='coerce')
df_thin['file_age_months'] = pd.to_numeric(df_thin['file_age_months'], errors='coerce')
filtered = df_thin[(df_thin['L3M_NRAK_PL_NOF_ENQURY'] > 0.5) & (df_thin['L3M_NRAK_NOF_ENQURY'] > 5.5) & (df_thin['file_age_months'] <= 4.50)].copy()
print(f"Total df_thin rows: {len(df_thin):,}")
print(f"Rows with L3M_NRAK_PL_NOF_ENQURY > 0.5: {len(filtered):,} ({len(filtered)/len(df_thin)*100:.2f}%)")
print()

# --- Full DELINQUENCYBUCKET distribution per snapshot_month -------------------
print("=" * 80)
print("DELINQUENCYBUCKET distribution per snapshot_month (L3M_NRAK_PL_NOF_ENQURY > 0.5)")
print("=" * 80)

cross = pd.crosstab(filtered['snapshot_month'], filtered['DELINQUENCYBUCKET'], margins=True)
display(cross)

# --- Focus on DELINQUENCYBUCKET > 3 ------------------------------------------
severe = filtered[filtered['DELINQUENCYBUCKET'] > 3].copy()
print(f"\nRows with DELINQUENCYBUCKET > 3: {len(severe):,} ({len(severe)/len(filtered)*100:.2f}% of filtered)")
print()

severe_by_month = (
    severe
    .groupby('snapshot_month')
    .agg(
        count=('DELINQUENCYBUCKET', 'count'),
        mean_bucket=('DELINQUENCYBUCKET', 'mean'),
        max_bucket=('DELINQUENCYBUCKET', 'max'),
        unique_cards=('CREDITCARDNO', 'nunique'),
    )
    .reset_index()
)

# Also compute total filtered per month for rate calculation
total_by_month = filtered.groupby('snapshot_month').size().reset_index(name='total_filtered')
severe_by_month = severe_by_month.merge(total_by_month, on='snapshot_month', how='left')
severe_by_month['severe_rate_pct'] = (severe_by_month['count'] / severe_by_month['total_filtered'] * 100).round(2)

print("DELINQUENCYBUCKET > 3 per snapshot_month:")
print("=" * 80)
display(severe_by_month)

# --- Plot: severe count + rate per snapshot_month -----------------------------
fig, ax1 = plt.subplots(figsize=(14, 6))
x_labels = severe_by_month['snapshot_month'].astype(str)
x_pos = range(len(x_labels))

ax1.bar(x_pos, severe_by_month['count'], color='#e74c3c', edgecolor='white', label='Count (Bucket > 3)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax1.set_xlabel('Snapshot Month', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('DELINQUENCYBUCKET > 3 per Snapshot Month\n(filtered: L3M_NRAK_PL_NOF_ENQURY > 0.5)', fontsize=13)
ax1.legend(loc='upper left', fontsize=10)

ax2 = ax1.twinx()
ax2.plot(x_pos, severe_by_month['severe_rate_pct'], color='navy', marker='o', linewidth=2, label='Severe Rate %')
ax2.set_ylabel('Severe Rate %', fontsize=12, color='navy')
ax2.tick_params(axis='y', labelcolor='navy')
ax2.legend(loc='upper right', fontsize=10)

ax1.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
'Max_CC_utilization' in aecb_pd.columns

# ============================================================================
# Filter df_thin where L3M_NRAK_PL_NOF_ENQURY > 0.5
# Check DELINQUENCYBUCKET distribution per snapshot_month
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt

# Ensure numeric types
df_thin['L3M_NRAK_PL_NOF_ENQURY'] = pd.to_numeric(df_thin['L3M_NRAK_PL_NOF_ENQURY'], errors='coerce')
df_thin['DELINQUENCYBUCKET'] = pd.to_numeric(df_thin['DELINQUENCYBUCKET'], errors='coerce')
df_thin['snapshot_month'] = pd.to_datetime(df_thin['SNAPSHOT_DATE']).dt.to_period('M')

# Filter: L3M_NRAK_PL_NOF_ENQURY > 0.5 AND L3M_NRAK_NOF_ENQURY > 5.5
df_thin['L3M_NRAK_NOF_ENQURY'] = pd.to_numeric(df_thin['L3M_NRAK_NOF_ENQURY'], errors='coerce')
df_thin['Max_CC_utilization'] = pd.to_numeric(df_thin['Max_CC_utilization'], errors='coerce')
filtered = df_thin[(df_thin['L3M_NRAK_PL_NOF_ENQURY'] > 0.5) & (df_thin['L3M_NRAK_NOF_ENQURY'] > 5.5) & (df_thin['Max_CC_utilization'] <= 2.37)].copy()
print(f"Total df_thin rows: {len(df_thin):,}")
print(f"Rows with L3M_NRAK_PL_NOF_ENQURY > 0.5: {len(filtered):,} ({len(filtered)/len(df_thin)*100:.2f}%)")
print()

# --- Full DELINQUENCYBUCKET distribution per snapshot_month -------------------
print("=" * 80)
print("DELINQUENCYBUCKET distribution per snapshot_month (L3M_NRAK_PL_NOF_ENQURY > 0.5)")
print("=" * 80)

cross = pd.crosstab(filtered['snapshot_month'], filtered['DELINQUENCYBUCKET'], margins=True)
display(cross)

# --- Focus on DELINQUENCYBUCKET > 3 ------------------------------------------
severe = filtered[filtered['DELINQUENCYBUCKET'] > 3].copy()
print(f"\nRows with DELINQUENCYBUCKET > 3: {len(severe):,} ({len(severe)/len(filtered)*100:.2f}% of filtered)")
print()

severe_by_month = (
    severe
    .groupby('snapshot_month')
    .agg(
        count=('DELINQUENCYBUCKET', 'count'),
        mean_bucket=('DELINQUENCYBUCKET', 'mean'),
        max_bucket=('DELINQUENCYBUCKET', 'max'),
        unique_cards=('CREDITCARDNO', 'nunique'),
    )
    .reset_index()
)

# Also compute total filtered per month for rate calculation
total_by_month = filtered.groupby('snapshot_month').size().reset_index(name='total_filtered')
severe_by_month = severe_by_month.merge(total_by_month, on='snapshot_month', how='left')
severe_by_month['severe_rate_pct'] = (severe_by_month['count'] / severe_by_month['total_filtered'] * 100).round(2)

print("DELINQUENCYBUCKET > 3 per snapshot_month:")
print("=" * 80)
display(severe_by_month)

# --- Plot: severe count + rate per snapshot_month -----------------------------
fig, ax1 = plt.subplots(figsize=(14, 6))
x_labels = severe_by_month['snapshot_month'].astype(str)
x_pos = range(len(x_labels))

ax1.bar(x_pos, severe_by_month['count'], color='#e74c3c', edgecolor='white', label='Count (Bucket > 3)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax1.set_xlabel('Snapshot Month', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('DELINQUENCYBUCKET > 3 per Snapshot Month\n(filtered: L3M_NRAK_PL_NOF_ENQURY > 0.5)', fontsize=13)
ax1.legend(loc='upper left', fontsize=10)

ax2 = ax1.twinx()
ax2.plot(x_pos, severe_by_month['severe_rate_pct'], color='navy', marker='o', linewidth=2, label='Severe Rate %')
ax2.set_ylabel('Severe Rate %', fontsize=12, color='navy')
ax2.tick_params(axis='y', labelcolor='navy')
ax2.legend(loc='upper right', fontsize=10)

ax1.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()